# flash-attention-residuals — Colab benchmark

Runs `benchmarks/bench.py` from the fork at `NotDevinLe/flash-attention-residuals`.

**Before running:** `Runtime → Change runtime type → GPU`. Default bench shapes (`B=4, T=16384, D=2048`) need a high-memory GPU — pick **A100** or at least **L4**. A T4 (16 GB) will OOM.

If you want to bench a private branch, edit `REPO_URL` / `BRANCH` in the next cell.

In [ ]:
REPO_URL = "https://github.com/NotDevinLe/flash-attention-residuals.git"
BRANCH = "main"
REPO_DIR = "flash-attention-residuals"

In [ ]:
# Confirm a GPU is attached and show its name / memory.
!nvidia-smi

In [ ]:
# Clone the fork (skip if already cloned in this session).
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest")
    !cd {REPO_DIR} && git pull --ff-only

In [ ]:
%cd {REPO_DIR}

In [ ]:
# Install the package editable + the bench-only dep (liger-kernel).
# Colab already ships with a CUDA-enabled torch + triton, so we reuse those.
!pip install -q -e .
!pip install -q liger-kernel

In [ ]:
# Sanity check the install.
import torch, triton, flash_attn_res
print("torch:  ", torch.__version__, "| cuda:", torch.version.cuda, "| device:", torch.cuda.get_device_name(0))
print("triton: ", triton.__version__)
print("pkg:    ", flash_attn_res.__file__)

In [ ]:
# Run the benchmark. bench.py sets TRITON_PRINT_AUTOTUNING=1 itself,
# so expect autotune logs before each kernel's first timed run.
!python benchmarks/bench.py